In [35]:
from aicsimageio import AICSImage
import dask_image.imread


In [33]:
path = "/Volumes/KINGSTON/code/phd/image-analysis/synapse-counting/test-images-VLGUT1-PSD95-A/OE_Exp1_IHC_Exp1_HA-GPR37L1_555-VGLUT1_647-PSD95_63X_airyscan_1.8zoom_CA1_SO.czi"

img = AICSImage(path)
img.dask_data 

dask.array<transpose, shape=(1, 2, 1, 1000, 1000), dtype=uint16, chunksize=(1, 1, 1, 1000, 1000), chunktype=numpy.ndarray>

In [33]:
import os
import pandas as pd
import dask

from metadata import extract_metadata, image_filename
from preprocessing import extract_and_split, ImagePreprocessing
from calc_synaptic_coloc import pearsons_coloc


In [34]:
input_folder = "/Volumes/KINGSTON/code/phd/image-analysis/synapse-counting/test-images-VLGUT1-PSD95-A"

In [50]:
@delayed
def process_image_pearson(filename):    
    if filename.endswith(".czi"):
        # getting the filepath
        file_path = os.path.join(input_folder, filename)
        # metadata
        pixel_size_um, image_size_pix, image_size_um = extract_metadata(file_path)
        # preprocessing
        pre, post = extract_and_split(file_path)
        p = ImagePreprocessing(include_rolling_ball=True, include_blur=True, include_clahe=True, include_tophat=True)
        pre_1, post_1 = p.preprocess(pre, post)
        # getting the data
        pearson_cor, pvalue, pearson_cor_rot, pvalue_rot = pearsons_coloc(pre_1, post_1)
        # getting the right filename
        img_filename = image_filename(filename, [1,2,3,4,11]) 
        return {
            "img_filename": img_filename,
            "pearson_cor": pearson_cor,
            "pvalue": pearson_cor_rot,
            "pearson_cor_rot": pearson_cor_rot,
            "pvalue_rot": pvalue_rot
        }
    else:
        return None

In [51]:
# get a list of files in that input_folder
file_list = os.listdir(input_folder)

In [52]:
delayed_results = [process_image_pearson(filename) for filename in file_list]

In [53]:
print(delayed_results)

[Delayed('process_image_pearson-024c2aee-0b17-4d3d-b738-97af8e1d5d70'), Delayed('process_image_pearson-f146e214-2373-494a-bd52-3d51a62bf89a'), Delayed('process_image_pearson-68c87c4b-edf5-42bc-97a1-2c13ee908f52'), Delayed('process_image_pearson-69dbe266-afbc-4f3f-aed6-6106c752f16e'), Delayed('process_image_pearson-a5e02175-c752-4b68-a742-86206c2972c4'), Delayed('process_image_pearson-955b6a35-f879-43c2-8b77-0e8527454849'), Delayed('process_image_pearson-83ae48ff-c5f8-47d2-bb36-0a6d5286212e'), Delayed('process_image_pearson-8d4dcc5d-c25d-4199-8ef6-ba17597c0e87'), Delayed('process_image_pearson-767b449a-aa9e-4033-aaf8-7d7afb2e2746'), Delayed('process_image_pearson-aae32709-605a-4b60-88a7-0eff9d4295f7'), Delayed('process_image_pearson-72e6c8ff-d7d1-4625-a7ae-bd90e06ace0f'), Delayed('process_image_pearson-d3d6f75c-6f2b-4546-b860-a67c63871ca8'), Delayed('process_image_pearson-0c23cb36-ef09-4ee9-8829-2d663d8b3d57'), Delayed('process_image_pearson-1a3f8526-fc88-4879-a6d8-e8dea6aadc12'), Delay

In [58]:
dask.visualize(*delayed_results)

CytoscapeWidget(cytoscape_layout={'name': 'dagre', 'rankDir': 'BT', 'nodeSep': 10, 'edgeSep': 10, 'spacingFact…

In [54]:
results = dask.compute(*delayed_results)

In [55]:
df = pd.DataFrame(results)
df.head(10)

,img_filename,pearson_cor,pvalue,pearson_cor_rot,pvalue_rot
0,Exp1_IHC_Exp1_HA-GPR37L1_SO,0.182199,-0.009077,-0.009077,1.111628e-19
1,Exp1_IHC_Exp1_HA-GPR37L1_SO,0.221573,-0.001514,-0.001514,1.299304e-01
2,Exp1_IHC_Exp1_HA-GPR37L1_SR,0.223146,0.003013,0.003013,2.589744e-03
3,Exp1_IHC_Exp1_HA-GPR37L1_SR,0.237378,-0.002722,-0.002722,6.491481e-03
4,Exp1_IHC_Exp1_HA-GPR37L1_SL,0.216331,0.012995,0.012995,1.302848e-38
5,Exp1_IHC_Exp1_HA-GPR37L1_SL,0.235871,0.003791,0.003791,1.502787e-04
6,Exp1_IHC_Exp1_HA-GPR37L1_SO,0.204238,0.003481,0.003481,4.999484e-04
7,Exp1_IHC_Exp1_HA-GPR37L1_SO,0.223105,-0.002297,-0.002297,2.159668e-02
8,Exp1_IHC_Exp1_HA-GPR37L1_SR,0.235140,0.000733,0.000733,4.632632e-01
9,Exp1_IHC_Exp1_HA-GPR37L1_SR,0.225649,-0.001705,-0.001705,8.820645e-02


In [20]:
# get a list of files in that input_folder
file_list = os.listdir(input_folder)

# empty dict to store the results
results = []

# the actual function
for filename in file_list:
    if filename.endswith(".czi"):
        
        # getting the filepath
        file_path = os.path.join(input_folder, filename)
        
        # metadata
        pixel_size_um, image_size_pix, image_size_um = extract_metadata(file_path)
        
        # preprocessing
        pre, post = extract_and_split(file_path)
        p = ImagePreprocessing(include_rolling_ball=True, include_blur=True, include_clahe=True, include_tophat=True)
        pre_1, post_1 = delayed(p.preprocess)(pre, post)
        
        # getting the data
        pearson_cor, pvalue, pearson_cor_rot, pvalue_rot = delayed(pearsons_coloc)(pre_1, post_1)
        
        # getting the right filename
        img_filename = delayed(image_filename)(filename, [1,2,3,4,11]) 

        results.append({
            "img_filename": img_filename,
            "pearson_cor": pearson_cor,
            "pvalue": pearson_cor_rot,
            "pearson_cor_rot": pearson_cor_rot,
            "pvalue_rot": pvalue_rot
        })

TypeError: Delayed objects of unspecified length are not iterable

In [14]:
df.head(30)

,img_filename,pearson_cor,pvalue,pearson_cor_rot,pvalue_rot
0,Exp1_IHC_Exp1_HA-GPR37L1_SO,0.182199,-0.009077,-0.009077,1.111628e-19
1,Exp1_IHC_Exp1_HA-GPR37L1_SO,0.221573,-0.001514,-0.001514,1.299304e-01
2,Exp1_IHC_Exp1_HA-GPR37L1_SR,0.223146,0.003013,0.003013,2.589744e-03
3,Exp1_IHC_Exp1_HA-GPR37L1_SR,0.237378,-0.002722,-0.002722,6.491481e-03
4,Exp1_IHC_Exp1_HA-GPR37L1_SL,0.216331,0.012995,0.012995,1.302848e-38
5,Exp1_IHC_Exp1_HA-GPR37L1_SL,0.235871,0.003791,0.003791,1.502787e-04
6,Exp1_IHC_Exp1_HA-GPR37L1_SO,0.204238,0.003481,0.003481,4.999484e-04
7,Exp1_IHC_Exp1_HA-GPR37L1_SO,0.223105,-0.002297,-0.002297,2.159668e-02
8,Exp1_IHC_Exp1_HA-GPR37L1_SR,0.235140,0.000733,0.000733,4.632632e-01
9,Exp1_IHC_Exp1_HA-GPR37L1_SR,0.225649,-0.001705,-0.001705,8.820645e-02
